# Base analítica: una fila por llamada

Convierte las transcripciones con rol asignado (`Transcriptions procesadas/transcriptions
con agente y deudor/`) en una tabla plana lista para comparar humanos vs. IA.

**Entrada:** llamadas humanas + de IA, con `rol` (AGENTE / DEUDOR) en cada segmento.
**Salida:** `Base analitica/base_analitica.csv` —  una por llamada.

Solo usa la librería estándar de Python: el resultado es reproducible sin instalar nada.

## Qué mide cada bloque de variables

Las variables aquí sustentan **cuatro hipótesis** (ver `informe/informe.tex` y `NOTAS.md`):
se descartaron variables que dependían de detectar palabras sueltas sin verificar el sentido
de la frase (empatía, "gates" de permiso, canal de pago) porque son poco robustas — el mismo
patrón puede aparecer con un significado distinto al que se busca medir.

| Bloque | Hipótesis que sustenta | Variables |
|---|---|---|
| **Ritmo conversacional** | H1: la IA responde más lento | latencia de respuesta, silencio, turnos, velocidad de habla |
| **Reparto de habla** | contexto para H1 | palabras y tiempo de voz por rol |
| **Presión / urgencia** | H2: la IA presiona más con lo legal | menciones a proceso legal, embargo, urgencia |
| **Protocolo de apertura** | H6: la IA cumple mejor el guion de apertura | se identifica, verifica identidad |
| **Calidad de la medición** | H7: la llamada de IA es más auditable | diarización colapsada, segmentos sin rol |

Las variables de resultado (compromiso de pago, tipo de objeción, manejo de la objeción) **no
están aquí**: son semánticas y se etiquetan aparte con un LLM, igual que se hizo con el rol
agente/deudor.

## Rutas

In [1]:
import json
import csv
import re
import statistics as st
from pathlib import Path

REPO_ROOT = Path(".").resolve()
if not (REPO_ROOT / "Transcriptions procesadas").exists():
    REPO_ROOT = REPO_ROOT.parent  # por si se corre desde /Scripts

BASE_DIR = REPO_ROOT / "Transcriptions procesadas" / "transcriptions con agente y deudor"
OUT_DIR = REPO_ROOT / "Base analitica"
OUT_DIR.mkdir(exist_ok=True)

print("Entrada:", BASE_DIR, "| existe:", BASE_DIR.exists())
print("Salida: ", OUT_DIR)

Entrada: C:\Users\andre\Desktop\Personal\Pruebas tecnicas\crecere\prueba_crecere\Transcriptions procesadas\transcriptions con agente y deudor | existe: True
Salida:  C:\Users\andre\Desktop\Personal\Pruebas tecnicas\crecere\prueba_crecere\Base analitica


## Cargar las llamadas

In [2]:
def cargar(grupo):
    """Lee todos los JSON de un grupo (humanos / ia)."""
    return [json.loads(p.read_text(encoding="utf-8"))
            for p in sorted((BASE_DIR / grupo).glob("*.json"))]


LLAMADAS = [("humano", c) for c in cargar("humanos")] + [("ia", c) for c in cargar("ia")]

print("Humanas:", sum(1 for g, _ in LLAMADAS if g == "humano"))
print("IA:     ", sum(1 for g, _ in LLAMADAS if g == "ia"))

Humanas: 48
IA:      49


## Léxicos

Solo quedan los dos patrones de texto que sustentan una hipótesis concreta y que se
verificaron leyendo transcripciones de los dos grupos:

- `presion`: consecuencia legal o urgencia artificial (H2).
- `identificacion` / `verificacion`: fórmulas de apertura del guion (H6).

Dos advertencias que condicionan la lectura:

- Los audios están **censurados con un pitido** sobre nombres propios y entidades, así que
  ningún patrón puede depender del nombre de la empresa. Por eso `identificacion` busca la
  *fórmula* de presentación (`le habla`, `le llamo de`), no la marca.
- Son patrones **léxicos**: detectan si la palabra aparece, no si el sentido de la frase
  coincide con lo que se busca medir. Es una limitación que se acepta a cambio de que el
  cálculo sea 100 % reproducible y verificable a mano.

In [3]:
PATRONES = {
    # Presión: consecuencia legal o urgencia artificial
    "presion": (r"proceso legal|proceso jur[ií]dic|embargo|embargar|judicial|demanda|abogado|"
                r"centrales de riesgo|reporte negativo|tiempo limitado|vence hoy|"
                r"[uú]ltima oportunidad|se vence"),
    # Protocolo de apertura (sin depender de nombres, que están censurados)
    "identificacion": (r"le habla|mi nombre es|me comunico|nos comunicamos|le llamo|lo llamo|"
                       r"la llamo|les llamo|estamos llamando|casa de cobranza|del [aá]rea de|"
                       r"en nombre de"),
    "verificacion": (r"confirmarme si (estoy|es)|hablo con|habla con|hablando con|"
                     r"por favor (la|el) se[ñn]or|me valide si es usted|"
                     r"confirmar (su|sus) (identidad|datos|nombre)|por confidencialidad|"
                     r"con qui[eé]n tengo"),
}

RX = {nombre: re.compile(patron) for nombre, patron in PATRONES.items()}
print(len(RX), "patrones compilados")

3 patrones compilados


## El turno como unidad de análisis

Whisper corta el audio en fragmentos cortos (a veces media frase). Medir sobre fragmentos
infla los conteos y distorsiona el ritmo. Antes de calcular nada, se agrupan los segmentos
consecutivos del mismo rol en un **turno**: una intervención continua de una persona.

Sobre los turnos se define la variable central de H1, la **latencia de respuesta**: los
segundos de silencio entre el final de un turno y el inicio de la respuesta del otro rol.
Se calcula para los dos roles a propósito: la del deudor funciona como **control**. Si el
deudor responde igual de rápido en los dos grupos, la lentitud de la IA es del agente y no
del canal telefónico.

In [4]:
def a_turnos(llamada):
    """Agrupa segmentos consecutivos del mismo rol en un solo turno."""
    turnos = []
    for seg in llamada["segments"]:
        if seg["rol"] not in ("AGENTE", "DEUDOR"):
            continue  # INCIERTO / UNKNOWN no se atribuyen a nadie
        if turnos and turnos[-1]["rol"] == seg["rol"]:
            actual = turnos[-1]
            actual["end"] = seg["end"]
            actual["texto"] += " " + seg["text"]
            actual["voz"] += seg["end"] - seg["start"]
        else:
            turnos.append({
                "rol": seg["rol"],
                "start": seg["start"],
                "end": seg["end"],
                "texto": seg["text"],
                "voz": seg["end"] - seg["start"],
            })
    return turnos


def latencias(turnos, rol_que_responde):
    """Silencios (s) antes de cada respuesta de `rol_que_responde`."""
    return [b["start"] - a["end"]
            for a, b in zip(turnos, turnos[1:])
            if b["rol"] == rol_que_responde and a["rol"] != rol_que_responde
            and b["start"] >= a["end"]]

## Funciones de apoyo

In [5]:
def mediana(valores):
    """Mediana, o vacío si no hay datos (mejor que un 0 que se confunde con un dato real)."""
    return round(st.median(valores), 3) if valores else ""


def cuenta(patron, texto):
    return len(RX[patron].findall(texto.lower()))


def hay(patron, texto):
    return int(bool(RX[patron].search(texto.lower())))

## Construcción de la fila

Criterios de normalización, para que las comparaciones no midan simplemente "quién habló más":

- Se usan **medianas** dentro de cada llamada: hay un outlier humano de 20 minutos que
  desplaza cualquier promedio.
- `identificacion` / `verificacion` se buscan solo en los primeros 5 turnos del agente: es lo
  que define el protocolo de apertura, no toda la llamada.

In [6]:
def construir_fila(grupo, llamada):
    turnos = a_turnos(llamada)
    t_agente = [t for t in turnos if t["rol"] == "AGENTE"]
    t_deudor = [t for t in turnos if t["rol"] == "DEUDOR"]

    txt_agente = " ".join(t["texto"] for t in t_agente)
    apertura = " ".join(t["texto"] for t in t_agente[:5])  # protocolo: primeros 5 turnos

    duracion = llamada["duration_sec"]
    pal_agente, pal_deudor = len(txt_agente.split()), len(" ".join(t["texto"] for t in t_deudor).split())
    voz_agente = sum(t["voz"] for t in t_agente)
    voz_deudor = sum(t["voz"] for t in t_deudor)
    voz_total = sum(s["end"] - s["start"] for s in llamada["segments"])

    sin_rol = sum(1 for s in llamada["segments"] if s["rol"] in ("INCIERTO", "UNKNOWN"))

    return {
        # --- identificación ---
        "short_id": llamada["short_id"],
        "grupo": grupo,
        "duracion_seg": duracion,

        # --- H1: ritmo conversacional ---
        "latencia_agente_med": mediana(latencias(turnos, "AGENTE")),
        "latencia_deudor_med": mediana(latencias(turnos, "DEUDOR")),
        "silencio_pct": round(max(0.0, 1 - voz_total / duracion), 3),
        "n_turnos": len(turnos),
        "turnos_por_min": round(len(turnos) / (duracion / 60), 2),
        "dur_turno_agente_med": mediana([t["voz"] for t in t_agente]),
        "dur_turno_deudor_med": mediana([t["voz"] for t in t_deudor]),
        "vel_agente_pps": round(pal_agente / voz_agente, 2) if voz_agente else "",
        "vel_deudor_pps": round(pal_deudor / voz_deudor, 2) if voz_deudor else "",

        # --- contexto de reparto de habla ---
        "palabras_agente": pal_agente,
        "palabras_deudor": pal_deudor,
        "share_tiempo_agente": (round(voz_agente / (voz_agente + voz_deudor), 3)
                                if voz_agente + voz_deudor else ""),

        # --- H2: presión / urgencia ---
        "n_presion_legal": cuenta("presion", txt_agente),
        "presiona": int(cuenta("presion", txt_agente) > 0),

        # --- H6: protocolo de apertura ---
        "se_identifica": hay("identificacion", apertura),
        "verifica_identidad": hay("verificacion", apertura),

        # --- H7 y calidad de la medición / auditabilidad ---
        "n_speakers_detected": llamada["n_speakers_detected"],
        "diarizacion_colapsada": int(llamada["n_speakers_detected"] == 1),
        "diarizacion_limpia": int(llamada["n_speakers_detected"] == 2 and sin_rol == 0),
        "pct_segmentos_sin_rol": (round(sin_rol / len(llamada["segments"]), 3)
                                  if llamada["segments"] else ""),
    }

## Generar y guardar

In [7]:
BASE = [construir_fila(grupo, llamada) for grupo, llamada in LLAMADAS]
COLUMNAS = list(BASE[0].keys())

csv_path = OUT_DIR / "base_analitica.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=COLUMNAS)
    writer.writeheader()
    writer.writerows(BASE)

assert len(BASE) == 97, f"Se esperaban 97 llamadas, hay {len(BASE)}"
print(f"{len(BASE)} llamadas x {len(COLUMNAS)} variables -> {csv_path.name}")

97 llamadas x 23 variables -> base_analitica.csv


## Resumen comparativo

Para variables continuas se muestra la **mediana** (robusta al outlier de 20 minutos); para
las binarias, el **porcentaje de llamadas donde el fenómeno aparece**.

Este cuadro es exploratorio: los contrastes formales (Mann-Whitney y proporciones) van en el
notebook de análisis.

In [8]:
CONTINUAS = [
    "duracion_seg", "latencia_agente_med", "latencia_deudor_med", "silencio_pct",
    "turnos_por_min", "dur_turno_agente_med", "dur_turno_deudor_med", "vel_agente_pps",
]
CONTEOS = ["n_presion_legal"]
BINARIAS = [
    "presiona", "se_identifica", "verifica_identidad",
    "diarizacion_colapsada", "diarizacion_limpia",
]

H = [r for r in BASE if r["grupo"] == "humano"]
IA = [r for r in BASE if r["grupo"] == "ia"]
vals = lambda filas, col: [float(r[col]) for r in filas if r[col] != ""]


def imprimir(titulo, columnas, resumen, encabezado):
    print(f"\n{titulo}")
    print(f"{'variable':<26}{encabezado[0]:>12}{encabezado[1]:>12}{'n h/ia':>10}")
    print("-" * 60)
    for col in columnas:
        h, i = vals(H, col), vals(IA, col)
        print(f"{col:<26}{resumen(h):>12}{resumen(i):>12}{f'{len(h)}/{len(i)}':>10}")


fmt = lambda x: f"{x:.2f}"
imprimir("MEDIANAS", CONTINUAS, lambda v: fmt(st.median(v)), ("humano", "IA"))
imprimir("CONTEOS (promedio por llamada)", CONTEOS, lambda v: fmt(st.mean(v)), ("humano", "IA"))
imprimir("PRESENCIA (% de llamadas)", BINARIAS,
         lambda v: f"{100 * st.mean(v):.0f}%", ("humano", "IA"))


MEDIANAS
variable                        humano          IA    n h/ia
------------------------------------------------------------
duracion_seg                    171.10      156.08     48/49
latencia_agente_med               0.27        1.87     43/40
latencia_deudor_med               0.11        0.86     44/41
silencio_pct                      0.16        0.22     48/49
turnos_por_min                    3.87        4.21     48/49
dur_turno_agente_med             15.29        9.57     48/45
dur_turno_deudor_med              2.92        2.83     44/41
vel_agente_pps                    2.62        2.80     48/45

CONTEOS (promedio por llamada)
variable                        humano          IA    n h/ia
------------------------------------------------------------
n_presion_legal                   1.00        3.00     48/49

PRESENCIA (% de llamadas)
variable                        humano          IA    n h/ia
------------------------------------------------------------
presiona        